# Symptom Annotation Tool

This notebook allows you to randomly sample 17 cases and annotate them for symptoms.
Progress is saved automatically to `gold_standard_trinary.csv`.

In [1]:
import pandas as pd
import os
import random
import glob
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- CONFIGURATION ---
DATA_DIR = r"..\Clean Transcripts"
OUTPUT_FILE = "gold_standard_trinary.csv"

# Target counts for sampling
TARGETS = {
    "RES": 4,  # Respiratory
    "CAR": 4,  # Cardiac
    "MSK": 4,  # Musculoskeletal
    "GAS": 4,  # Gastrointestinal
    "DER": 4   # Dermatological (Note: Code will take max available if < 4)
}

# List of symptoms to annotate
SYMPTOMS = [
    "Fever", "Cough", "Shortness of Breath", "Sore Throat", "Chest Pain", 
    "Fatigue", "Headache", "Nausea", "Vomiting", "Diarrhea", 
    "Abdominal Pain", "Rash", "Joint Pain", "Muscle Pain", "Dizziness"
]
# You can add more symptoms to this list as needed

print("Setup complete.")

Setup complete.


In [2]:
def get_sampled_cases(data_dir, targets, output_file):
    # 1. If output file exists, load it to preserve existing samples/annotations
    if os.path.exists(output_file):
        print(f"Loading existing progress from {output_file}...")
        df = pd.read_csv(output_file)
        # Ensure all symptom columns exist
        for s in SYMPTOMS:
            if s not in df.columns:
                df[s] = 0 # Default to 0 (Not Present)
        return df

    # 2. Otherwise, perform new random sampling
    print("No existing file found. performing new random sampling...")
    sampled_files = []
    
    all_files = glob.glob(os.path.join(data_dir, "*.txt"))
    file_map = {}
    
    # Group files by prefix
    for f in all_files:
        basename = os.path.basename(f)
        prefix = basename[:3]
        if prefix not in file_map:
            file_map[prefix] = []
        file_map[prefix].append(f)
        
    # Sample according to targets
    records = []
    for prefix, count in targets.items():
        available = file_map.get(prefix, [])
        if len(available) < count:
            print(f"Warning: Requested {count} for {prefix}, but only found {len(available)}. Taking all.")
            selection = available
        else:
            selection = random.sample(available, count)
            
        for path in selection:
            filename = os.path.basename(path)
            records.append({
                "filename": filename,
                "category": prefix,
                "full_path": path
            })
            
    # Create DataFrame
    df = pd.DataFrame(records)
    # Initialize symptoms columns with 0 (Not Present)
    for s in SYMPTOMS:
        df[s] = 0
        
    # Save initial version
    df.to_csv(output_file, index=False)
    print(f"Sampled {len(df)} cases. Initialized {output_file}.")
    return df

In [3]:
class AnnotationWidget:
    def __init__(self, df, output_file):
        self.df = df
        self.output_file = output_file
        self.index = 0
        self.total_cases = len(df)
        
        # UI Elements
        self.output_area = widgets.Output()
        self.transcript_area = widgets.Textarea(
            disabled=True,
            layout=widgets.Layout(width='50%', height='500px')
        )
        
        self.symptom_widgets = {}
        symptom_vbox_children = []
        
        for s in SYMPTOMS:
            # 0: Not Present, 1: Negated, 2: Present
            w = widgets.ToggleButtons(
                options=[('Not Present', 0), ('Negated', 1), ('Present', 2)],
                description=s,
                style={'description_width': 'initial'},
                layout=widgets.Layout(margin='5px')
            )
            # Update value on change
            w.observe(lambda change, s=s: self.update_value(s, change['new']), names='value')
            self.symptom_widgets[s] = w
            symptom_vbox_children.append(w)
            
        self.controls_area = widgets.VBox(symptom_vbox_children, layout=widgets.Layout(width='45%', height='500px', overflow='auto'))
        
        # Navigation
        self.btn_prev = widgets.Button(description="<< Previous")
        self.btn_next = widgets.Button(description="Next >>")
        self.btn_save = widgets.Button(description="Force Save", button_style='success')
        
        self.btn_prev.on_click(self.on_prev)
        self.btn_next.on_click(self.on_next)
        self.btn_save.on_click(self.save_csv)
        
        self.nav_box = widgets.HBox([self.btn_prev, self.btn_next, self.btn_save])
        self.info_lbl = widgets.HTML()
        
        self.main_layout = widgets.VBox([
            self.info_lbl,
            widgets.HBox([self.transcript_area, self.controls_area]),
            self.nav_box,
            self.output_area
        ])
        
        self.load_case()

    def load_case(self):
        row = self.df.iloc[self.index]
        filename = row['filename']
        category = row['category']
        
        # Try to locate full path (reconstruct if needed)
        # If loading from CSV, 'full_path' might be missing or stale if moved.
        # Ideally we kept it, but let's be robust.
        if 'full_path' in row and os.path.exists(row['full_path']):
            path = row['full_path']
        else:
            # Fallback reconstruction
            path = os.path.join(DATA_DIR, filename)
            
        content = "[File not found]"
        if os.path.exists(path):
            with open(path, 'r', encoding='utf-8') as f:
                content = f.read()
        
        self.transcript_area.value = content
        self.info_lbl.value = f"<h2>Case {self.index + 1} / {self.total_cases}: {filename} ({category})</h2>"
        
        # Set widget values
        for s, w in self.symptom_widgets.items():
            val = row.get(s, 0)
            # Handle NaN or missing
            if pd.isna(val):
                val = 0
            w.value = int(val)

    def update_value(self, symptom, value):
        self.df.at[self.index, symptom] = value
        # Auto-save on every click could be slow, but safer? 
        # Let's auto-save on navigation instead, or lazy save.
        
    def save_csv(self, _=None):
        self.df.to_csv(self.output_file, index=False)
        with self.output_area:
            clear_output(wait=True)
            print(f"Saved to {self.output_file}")

    def on_prev(self, b):
        self.save_csv()
        if self.index > 0:
            self.index -= 1
            self.load_case()

    def on_next(self, b):
        self.save_csv()
        if self.index < self.total_cases - 1:
            self.index += 1
            self.load_case()

    def display(self):
        display(self.main_layout)

In [ ]:
# Run the Tool
df_cases = get_sampled_cases(DATA_DIR, TARGETS, OUTPUT_FILE)
app = AnnotationWidget(df_cases, OUTPUT_FILE)
app.display()

Loading existing progress from gold_standard_trinary.csv...
